In [1]:
%%capture
!pip install ydata-profiling


In [2]:
%%capture
!pip install torchsampler


In [3]:
%%capture
!pip install torchmetrics


In [4]:
%%capture
!pip install ray


In [5]:
import random
import numpy as np 
import pandas as pd 
import os
import datetime
import seaborn as sns
from tqdm.notebook import tqdm
from ydata_profiling import ProfileReport
from collections import Counter

from imblearn.over_sampling import SMOTE,SMOTEN
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve

from torchsampler import ImbalancedDatasetSampler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader,TensorDataset,random_split,SubsetRandomSampler, ConcatDataset
from torch.nn import functional as F
from torchmetrics import ROC
from torchmetrics.classification import BinaryAccuracy,BinaryROC


import ray
from ray import tune
from ray.air import session
from ray.air.checkpoint import Checkpoint
from ray.tune.schedulers import ASHAScheduler


import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('fivethirtyeight')
import plotly.express as px

import warnings
warnings.filterwarnings('ignore')
import itertools


ModuleNotFoundError: No module named 'sklearn.utils._metadata_requests'

In [6]:
for dirname, _, filenames in os.walk('/kaggle/input/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/train.csv
/kaggle/input/description.md
/kaggle/input/sample_submission.csv.zip
/kaggle/input/test.csv.zip
/kaggle/input/train.csv.zip
/kaggle/input/test.csv
/kaggle/input/sample_submission.csv
/kaggle/input/playground-series-s3e18/train.csv
/kaggle/input/playground-series-s3e18/description.md
/kaggle/input/playground-series-s3e18/sample_submission.csv.zip
/kaggle/input/playground-series-s3e18/test.csv.zip
/kaggle/input/playground-series-s3e18/train.csv.zip
/kaggle/input/playground-series-s3e18/test.csv
/kaggle/input/playground-series-s3e18/sample_submission.csv


In [7]:
class Datapreparation(object):
    
    def __init__(self,root_path):
        self.root_path = root_path
        
    def get_dataframe(self,filename):
        return pd.read_csv(os.path.join(self.root_path,filename))
    
    def summary(self,text, df):
        summary = pd.DataFrame(df.dtypes, columns=['dtypes'])
        summary['null'] = df.isnull().sum()
        summary['unique'] = df.nunique()
        summary['min'] = df.min()
        summary['median'] = df.median()
        summary['max'] = df.max()
        summary['mean'] = df.mean()
        summary['std'] = df.std()
        summary['duplicate'] = df.duplicated().sum()
        return summary
    
    
    def random_split_data(self,X,y):
        return train_test_split(X, y,test_size=0.20,random_state=42)

 
    def standardization_data(self,X_data):
        scaler = StandardScaler()
        std_X_data = scaler.fit_transform(X_data)
        return std_X_data
    

    
data = Datapreparation('/kaggle/input/playground-series-s3e18')
train=data.get_dataframe('train.csv')


In [8]:
data.summary('train',train)


,dtypes,null,unique,min,median,max,mean,std,duplicate
id,int64,0,13354,0.000000,7424.500000,14837.000000,7430.046054,4280.726674,0
BertzCT,float64,0,2234,0.000000,292.455280,4069.959780,515.344063,541.590240,0
Chi1,float64,0,1198,0.000000,6.494089,69.551167,9.135106,6.811711,0
Chi1n,float64,0,2976,0.000000,4.059093,50.174588,5.854336,4.641652,0
Chi1v,float64,0,3111,0.000000,4.392859,53.431954,6.741385,5.859587,0
Chi2n,float64,0,3385,0.000000,2.970427,32.195368,4.434309,3.759906,0
Chi2v,float64,0,3484,0.000000,3.261677,34.579313,5.251854,4.914257,0
Chi3v,float64,0,3223,0.000000,1.951087,22.589276,3.416554,3.425819,0
Chi4n,float64,0,2752,0.000000,1.077096,16.072810,1.772391,1.860338,0
EState_VSA1,float64,0,685,0.000000,17.353601,363.705954,29.207586,31.755738,0


In [9]:
#profile = ProfileReport(train, title="Train Data Profiling Report")
#profile


In [10]:
y1 = train['EC1']
y2 = train['EC2']
train.drop(columns=['id','EC1','EC2','EC3','EC4','EC5','EC6',
                    'BertzCT', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n', 'Chi2v', 'Chi3v', 'Chi4n'],axis=1,inplace=True)
X = train.copy()


In [11]:
X_train,X_val,y1_train,y1_val = data.random_split_data(X,y1)
print('Data splits for EC1 \n',X_train.shape,X_val.shape,y1_train.shape,y1_val.shape)


NameError: name 'train_test_split' is not defined

In [12]:
X_train,X_val,y2_train,y2_val = data.random_split_data(X,y2)
print('Data splits for EC2 \n',X_train.shape,X_val.shape,y2_train.shape,y2_val.shape)


NameError: name 'train_test_split' is not defined

In [13]:
std_X_train = data.standardization_data(X_train)
std_X_val = data.standardization_data(X_val)
print(std_X_train[0],std_X_val[0])


NameError: name 'X_train' is not defined

In [14]:
class Tensoroperations():
    
    def __init__(self):
        super(Tensoroperations,self).__init__()
    
    def convert_to_tensor(self,X,y=None):
        X_tensor =  torch.from_numpy(X).float()   
        y_tensor = torch.from_numpy(y).float() 
        return X_tensor,y_tensor
        
    def convert_to_test_tensor(self,X):
        X_tensor =  torch.from_numpy(X).float()
        return X_tensor
    
    def get_dataloaders(self,train_dataset,val_dataset):
        train_loaders = DataLoader(train_dataset,batch_size=32,shuffle=True)
        val_loaders = DataLoader(val_dataset,batch_size=32)
        return train_loaders,val_loaders
    
    def get_test_dataloaders(self,test_dataset,X_test):
        test_loaders = DataLoader(test_dataset,batch_size=X_test.shape[0])
        return test_loaders
        
        
    
tenops = Tensoroperations()


In [15]:
class CustomDataset(Dataset):
    
    def __init__(self,X_data,y_data=None,is_train=True):
        super().__init__()
        if is_train:
            self.X_data = X_data
            self.y_data = y_data
        else:
            self.X_data=X_train
            
    def __getitem__(self,index):
        return (self.X_data[index],self.y_data[index])
    
    def __len__(self):
        return len(self.X_data)


NameError: name 'Dataset' is not defined

In [16]:
X_tensor_train,y1_tensor_train = tenops.convert_to_tensor(std_X_train,y1_train.values)
X_tensor_val,y1_tensor_val = tenops.convert_to_tensor(std_X_val,y1_val.values)
print('The training tensor for EC1\n',X_tensor_train,y1_tensor_train)
print('The validation tensor for EC1\n',X_tensor_val,y1_tensor_val)


NameError: name 'std_X_train' is not defined

In [17]:
X_tensor_train,y2_tensor_train = tenops.convert_to_tensor(std_X_train,y2_train.values)
X_tensor_val,y2_tensor_val = tenops.convert_to_tensor(std_X_val,y2_val.values)
print('The training tensor EC2\n',X_tensor_train,y2_tensor_train)
print('The validation tensor Ec2\n',X_tensor_val,y2_tensor_val)


NameError: name 'std_X_train' is not defined

In [18]:
false_pos,true_pos = [],[]


In [19]:
class EnzymeClassificationBase(torch.nn.Module):
    
    def _accuracy(self,outputs, labels):
        #return torch.tensor(outputs.round() == labels).float().mean()
        metric = BinaryAccuracy()
        return metric(outputs, labels.unsqueeze(1))
    
    
    def training_step(self,batch):
        features,labels = batch
        out = self(features)
        loss = F.binary_cross_entropy(out,labels.unsqueeze(1))
        return loss
    
    def validation_step(self, batch):
        features, labels = batch 
        out = self(features)                    # Generate predictions
        loss = F.binary_cross_entropy(out, labels.unsqueeze(1))   # Calculate loss
        acc = self._accuracy(out, labels)           # Calculate accuracy
        self._get_roc(out, labels)
        return {'Validation_loss': loss.detach(), 'Validation_acc': acc}
    
    def _get_roc(self,output,labels):
        roc = ROC(task="binary")
        fpr, tpr, thresholds = roc(output, (labels.long()).unsqueeze(1))
        false_pos.append(fpr)
        true_pos.append(tpr)
        
        
    def validation_epoch_end(self, outputs):
        batch_losses = [x['Validation_loss'] for x in outputs]
        epoch_loss = torch.stack(batch_losses).mean()   # Combine losses
        batch_accs = [x['Validation_acc'] for x in outputs]
        epoch_acc = torch.stack(batch_accs).mean()      # Combine accuracies
        return {'Validation_loss': epoch_loss.item(), 'Validation_acc': epoch_acc.item()}
    
    def epoch_end(self, epoch, result):
        if epoch%10 == 0:
            print("Epoch [{}], Train_loss: {:.4f}, Validation_loss: {:.4f}, Validation_acc: {:.4f}".format(
                epoch, result['Train_loss'], result['Validation_loss'], result['Validation_acc']))


NameError: name 'torch' is not defined

In [20]:
n_input_dim = X_train.shape[1]
n_output =  1   # Number of output nodes = for binary classifier

class MultiClassificationNN(EnzymeClassificationBase):
    
    def __init__(self):
        super(MultiClassificationNN, self).__init__()
        
        self.network = torch.nn.Sequential(
            torch.nn.Linear(n_input_dim,32),
            torch.nn.LeakyReLU(),
            torch.nn.Linear(32,16),
            torch.nn.LeakyReLU(),
            torch.nn.Linear(16,8),
            torch.nn.LeakyReLU(),
            torch.nn.Linear(8,n_output),
            torch.nn.Sigmoid()
        )
    
    def forward(self, xb):
        return self.network(xb)


model_EC1 = MultiClassificationNN()
model_EC2 = MultiClassificationNN()


NameError: name 'X_train' is not defined

In [21]:
print(model_EC1)


NameError: name 'model_EC1' is not defined

In [22]:
print(model_EC2)


NameError: name 'model_EC2' is not defined

In [23]:
train1_dataset = CustomDataset(X_tensor_train,y1_tensor_train)
val1_dataset = CustomDataset(X_tensor_val,y1_tensor_val)

train2_dataset = CustomDataset(X_tensor_train,y2_tensor_train)
val2_dataset = CustomDataset(X_tensor_val,y2_tensor_val)


NameError: name 'CustomDataset' is not defined

In [24]:
train_dataloader = DataLoader(train1_dataset,64,shuffle=True)
val_dataloader = DataLoader(val1_dataset,64)


NameError: name 'DataLoader' is not defined

In [25]:
train_dataloader2 = DataLoader(train2_dataset,64,shuffle=True)
val_dataloader2 = DataLoader(val2_dataset,64)


NameError: name 'DataLoader' is not defined

In [26]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)


NameError: name 'torch' is not defined

In [27]:
class Trainer:
    
    @torch.no_grad()
    def _evaluate(self,model, val_loader):
        model.eval()
        outputs = [model.validation_step(batch) for batch in val_loader]
        return model.validation_epoch_end(outputs)

  
    def fit(self,epochs, lr, model, train_loader, val_loader, opt_func):
    
        history = []
        optimizer = opt_func(model.parameters(),lr,weight_decay=2e-5)
        for epoch in tqdm(range(epochs)):
        
            model.train()
            train_losses = []
            for batch in train_loader:
                loss = model.training_step(batch)
                train_losses.append(loss)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
            
            result = self._evaluate(model, val_loader)
            result['Train_loss'] = torch.stack(train_losses).mean().item()
            model.epoch_end(epoch, result)
            history.append(result)
    
        return history


NameError: name 'torch' is not defined

In [28]:
num_epochs=100
lr=1e-4
opt_func= torch.optim.Adam


NameError: name 'torch' is not defined

In [29]:
tariner=Trainer()
history_EC1 = tariner.fit(num_epochs, lr, model_EC1, train_dataloader, val_dataloader, opt_func)


NameError: name 'Trainer' is not defined

In [30]:
def plot_accuracies(history_EC1):
    """ Plot the history of accuracies"""
    accuracies = [x['Validation_acc'] for x in history_EC1]
    plt.plot(accuracies, '-x')
    plt.xlabel('Epoch')
    plt.ylabel('accuracy')
    plt.title('EC1 Accuracy vs. No. of epochs');
    
plot_accuracies(history_EC1)


NameError: name 'history_EC1' is not defined

In [31]:
def plot_losses(history_EC1):
    """ Plot the losses in each epoch"""
    train_losses = [x.get('Train_loss') for x in history_EC1]
    val_losses = [x['Validation_loss'] for x in history_EC1]
    plt.plot(train_losses, '-bx')
    plt.plot(val_losses, '-rx')
    plt.xlabel('Epoch')
    plt.ylabel('loss')
    plt.legend(['Training', 'Validation'])
    plt.title('EC1 Loss vs. No. of Epochs')

plot_losses(history_EC1)


NameError: name 'history_EC1' is not defined

In [32]:
fpr,tpr =[],[]
for i in range(len(false_pos)):
        fpr.append(np.mean(false_pos[i].tolist()))

for j in range(len(true_pos)):
        tpr.append(np.mean(false_pos[j].tolist()))


In [33]:
with torch.no_grad():
    plt.plot(fpr,tpr) # ROC curve = TPR vs FPR
    plt.title("Receiver Operating Characteristics")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.show()


NameError: name 'torch' is not defined

In [34]:
history_EC2 = tariner.fit(num_epochs, lr, model_EC2, train_dataloader2, val_dataloader2, opt_func)


NameError: name 'tariner' is not defined

In [35]:
def plot_accuracies(history_EC2):
    """ Plot the history of accuracies"""
    accuracies = [x['Validation_acc'] for x in history_EC2]
    plt.plot(accuracies, '-x')
    plt.xlabel('Epoch')
    plt.ylabel('accuracy')
    plt.title('EC2 Accuracy vs. No. of epochs');
    
plot_accuracies(history_EC2)


NameError: name 'history_EC2' is not defined

In [36]:
def plot_losses(history_EC2):
    """ Plot the losses in each epoch"""
    train_losses = [x.get('Train_loss') for x in history_EC2]
    val_losses = [x['Validation_loss'] for x in history_EC2]
    plt.plot(train_losses, '-bx')
    plt.plot(val_losses, '-rx')
    plt.xlabel('Epoch')
    plt.ylabel('loss')
    plt.legend(['Training', 'Validation'])
    plt.title('EC2 Loss vs. No. of Epochs')

plot_losses(history_EC2)


NameError: name 'history_EC2' is not defined

In [37]:
fpr,tpr =[],[]
for i in range(len(false_pos)):
        fpr.append(np.mean(false_pos[i].tolist()))

for j in range(len(true_pos)):
        tpr.append(np.mean(false_pos[j].tolist()))


In [38]:
with torch.no_grad():
    plt.plot(fpr, tpr) # ROC curve = TPR vs FPR
    plt.title("Receiver Operating Characteristics")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.show()


NameError: name 'torch' is not defined

In [39]:
test=data.get_dataframe('test.csv')


In [40]:
test.drop(columns=['BertzCT', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n', 'Chi2v', 'Chi3v', 'Chi4n'],axis=1,inplace=True)
data.summary('test',test)


,dtypes,null,unique,min,median,max,mean,std,duplicate
id,int64,0,1484,9.000000,7339.000000,14811.000000,7314.601078,4308.493169,0
EState_VSA1,float64,0,274,0.000000,17.488262,197.685072,29.159962,31.494745,0
EState_VSA2,float64,0,167,0.000000,6.286161,71.400084,10.210598,13.638512,0
ExactMolWt,float64,0,572,16.018724,204.089878,1306.602778,292.206410,228.227047,0
FpDensityMorgan1,float64,0,232,-14.458462,1.250000,3.000000,1.273746,0.540472,0
FpDensityMorgan2,float64,0,250,-14.458462,1.857143,3.111111,1.844263,0.542258,0
FpDensityMorgan3,float64,0,265,-14.458462,2.350877,3.125000,2.278709,0.582878,0
HallKierAlpha,float64,0,228,-7.730000,-1.100000,0.360000,-1.202187,0.938897,0
HeavyAtomMolWt,float64,0,418,14.007000,192.133000,1210.606000,274.743821,214.742675,0
Kappa3,float64,0,625,-104.040000,3.266728,24.919708,4.062419,8.769378,0


In [41]:
#profile_test = ProfileReport(test, title="Test Data Profiling Report")
#profile_test


In [42]:
test_update = test.loc[:, test.columns != 'id']
std_X_test = data.standardization_data(test_update)
std_X_test[0]


NameError: name 'StandardScaler' is not defined

In [43]:
X_tensor_test = tenops.convert_to_test_tensor(std_X_test)
X_tensor_test[0]


NameError: name 'std_X_test' is not defined

In [44]:
class CustomDataTest(Dataset):
    def __init__(self, X_data):
        self.X_data = X_data

        
    def __getitem__(self, index):
            return self.X_data[index]
        
    def __len__ (self):
        return len(self.X_data)


NameError: name 'Dataset' is not defined

In [45]:
test_dataset = CustomDataTest(X_tensor_test)
test_dataloader = DataLoader(test_dataset,64)


NameError: name 'CustomDataTest' is not defined

In [46]:
class Evaluate:
        
    def eval_test_data(self,model,test_data_dl):
        age_target = []
        model.eval()
        with torch.no_grad():
            for X_batch_test in test_data_dl:
                X_batch_test = X_batch_test.to(device)
                y_test_pred = model(X_batch_test)
                y_pred_tag = torch.sigmoid(y_test_pred)
                age_target.append(y_pred_tag.cpu())
        return [a.squeeze().tolist() for a in age_target]
    
    
eva = Evaluate()


In [47]:
from collections.abc import Iterable
def flatten(lis):
    for item in lis:
        if isinstance(item, Iterable) and not isinstance(item, str):
            for x in flatten(item):
                yield x
        else:        
            yield item
            
ec1 = eva.eval_test_data(model_EC1,test_dataloader)
ec2 = eva.eval_test_data(model_EC2,test_dataloader)


NameError: name 'model_EC1' is not defined

In [48]:
class Submit:
    
    def submit_predictions(self):        
        df_submit = pd.DataFrame(data={'id': test['id'],'EC1':  list(flatten(ec1)),'EC2':list(flatten(ec2))})
        df_submit.to_csv('submission.csv',index=False)
        print('Submission Completed!!')
        return df_submit
        
        
submit = Submit()
df_submit=submit.submit_predictions()


NameError: name 'ec1' is not defined

In [49]:
df_submit


NameError: name 'df_submit' is not defined